## Final benchmark construction

Use every eligible product under the declared source policy. Assign conservative complete catalog groups before cases and partitions. FDA chemical identifiers support ingredient comparisons. Missing identifiers cannot establish disjoint chemical sets. Freeze final_v5 and preserve all previously protected test products.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
assert (project_root / "src/hrp").is_dir()
source_dir = Path(os.environ.get("HRP_SOURCE_DIR", str(project_root.parent / "data")))

from hrp.benchmark import build_benchmark

summary = build_benchmark(project_root)
assert summary["split_frozen"]
assert all(summary["checks"][key] for key in (
    "all_accepted_products_used", "no_product_leakage", "no_family_leakage",
    "no_repeated_case_leakage", "reference_labels_before_evaluation",
))
assert summary["checks"]["test_inference_run"] is False
print(json.dumps({key: summary[key] for key in (
    "usable_products", "independent_family_groups", "benchmark_cases", "legacy_reservoir",
)}, indent=2))


Assigning all catalog products to conservative family groups


Family groups 20944


Constructing cases from all eligible products and 48741 comparison pairs


{
  "version": "final_v5",
  "status": "FROZEN",
  "usable_products": 53562,
  "independent_family_groups": 19984,
  "benchmark_cases": 145401,
  "partitions": {
    "train": {
      "cases": 81251,
      "cases_by_task": {
        "ambiguity_detection": 18656,
        "ingredient_comparison": 21421,
        "product_identification": 41174
      },
      "cases_by_type": {
        "different_ingredients_disjoint": 5586,
        "different_ingredients_shared": 3620,
        "explicit_ndc": 32227,
        "formulation_description": 8947,
        "name_only_ambiguous": 930,
        "name_only_identifiable": 17726,
        "same_ingredient_control": 12215
      },
      "usable_products": 32227,
      "independent_families": 13935,
      "families_by_task": {
        "ambiguity_detection": 12751,
        "ingredient_comparison": 3706,
        "product_identification": 13935
      },
      "families_by_type": {
        "different_ingredients_disjoint": 771,
        "different_ingredients_sh

{
  "usable_products": 53562,
  "independent_family_groups": 19984,
  "benchmark_cases": 145401,
  "legacy_reservoir": {
    "pairs": 1351,
    "products": 1713,
    "old_provisional_blocks": 583,
    "outcomes": {
      "eligible_test": 357,
      "eligible_train": 578,
      "eligible_validation": 84,
      "cross_partition": 330,
      "ingredient_evidence_unresolved": 2
    }
  }
}


In [2]:
display(pd.DataFrame({part: values["cases_by_task"] for part, values in summary["partitions"].items()}).fillna(0).astype(int))
display(pd.DataFrame({part: {key: values[key] for key in ("usable_products", "independent_families")}
                      for part, values in summary["partitions"].items()}))
print(json.dumps(summary["pair_construction_policy"], indent=2))
print(json.dumps(summary["construction_exclusions"], indent=2))
print("Pending audit sample", summary["review_queue"]["pending_sample"])


,train,validation,test
ambiguity_detection,18656,3826,6605
ingredient_comparison,21421,3631,23689
product_identification,41174,7936,18463


,train,validation,test
usable_products,32227,6185,15150
independent_families,13935,3013,3036


{
  "ingredient_eligible_products_considered": 53562,
  "chemical_comparison_eligible_products_considered": 53146,
  "distinct_name_formulation_prototypes": 41794,
  "possible_within_group_prototype_pairs": 2149854,
  "selected_unique_pairs": 48741,
  "selected_within_group_pairs": 48044,
  "cross_group_saved_pairs": 697,
  "not_enumerated_by_covering_policy": 2101810,
  "saved_candidate_outcomes": {
    "source_eligible_pair_considered": 2400,
    "cross_partition": 777,
    "same_prototype": 20,
    "source_or_chemical_identity_ineligible_endpoint": 5
  },
  "policy": "Collapse identical name and formulation replicas. Within each family connect every prototype in a same-ingredient cycle, to cyclic next shared and disjoint ingredient sets when available, and to adjacent sorted formulations. Include every eligible saved candidate pair within a partition. No family case cap. The covering graph avoids quadratic enumeration and repeated equivalent pairs."
}
{
  "duplicate_task_input": 331

The frozen test file has its own hash and is excluded from development evaluation. Training and validation may use the same catalog retrieval evidence that will later be supplied to Gemma. Repeated products and pairs do not increase the number of independent family groups. Original benchmark and retrieval results remain at their historical paths.
